In [6]:
import pandas as pd
import numpy as np

def answer_one():
    # 1. Energy
    energy = pd.read_excel('D:/Energy Indicators.xls', skiprows=17, skipfooter=38, engine='xlrd')
    energy = energy.iloc[:, 2:]
    energy.columns = ['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable']
    energy.replace('...', np.nan, inplace=True)
    energy['Energy Supply'] = pd.to_numeric(energy['Energy Supply']) * 1000000
    
    def clean_country(data):
        data = ''.join([i for i in data if not i.isdigit()])
        if '(' in data: data = data.split('(')[0]
        return data.strip()
    energy['Country'] = energy['Country'].apply(clean_country)
    energy['Country'] = energy['Country'].replace({
        "Republic of Korea": "South Korea", "United States of America": "United States",
        "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
        "China, Hong Kong Special Administrative Region": "Hong Kong"
    })

    # 2. GDP
    raw_gdp = pd.read_excel('D:/world_bank.xls', sheet_name='Data', engine='xlrd')
    h_row = 0
    for i in range(len(raw_gdp)):
        if "Country Name" in raw_gdp.iloc[i].values:
            h_row = i + 1
            break
    gdp = pd.read_excel('D:/world_bank.xls', sheet_name='Data', skiprows=h_row, engine='xlrd')
    gdp.rename(columns={gdp.columns[0]: 'Country'}, inplace=True)
    gdp['Country'] = gdp['Country'].replace({
        "Korea, Rep.": "South Korea", "Iran, Islamic Rep.": "Iran", "Hong Kong SAR, China": "Hong Kong"
    })
    gdp.columns = [str(c).split('.')[0] for c in gdp.columns]
    years = [str(y) for y in range(2006, 2016)]
    GDP_subset = gdp[['Country'] + years]

    # 3. ScimEn
    ScimEn = pd.read_excel('D:/scimagojr.xlsx', engine='openpyxl')
    ScimEn_top15 = ScimEn[:15]

    # Merge
    df = pd.merge(ScimEn_top15, energy, how='inner', on='Country')
    final_df = pd.merge(df, GDP_subset, how='inner', on='Country')
    
    # Видаляємо зайву колонку Region, щоб було рівно 20 колонок, як у завданні
    if 'Region' in final_df.columns:
        final_df = final_df.drop('Region', axis=1)
        
    return final_df.set_index('Country')

def answer_two():
    Top15 = answer_one()
    years = [str(y) for y in range(2006, 2016)]
    avgGDP = Top15[years].mean(axis=1).sort_values(ascending=False)
    return avgGDP.rename('avgGDP')

def answer_three():
    Top15 = answer_one()
    avgGDP = answer_two()
    # 6-та країна в списку (індекс 5)
    country = avgGDP.index[5]
    return float(Top15.loc[country, '2015'] - Top15.loc[country, '2006'])

def answer_four():
    Top15 = answer_one()
    Top15['Ratio'] = Top15['Self-citations'] / Top15['Citations']
    return (Top15['Ratio'].idxmax(), Top15['Ratio'].max())

def answer_five():
    Top15 = answer_one()
    Top15['PopEst'] = Top15['Energy Supply'] / Top15['Energy Supply per Capita']
    # Повертаємо назву 3-ї країни
    return str(Top15['PopEst'].sort_values(ascending=False).index[2])

def answer_six():
    Top15 = answer_one()
    Top15['PopEst'] = Top15['Energy Supply'] / Top15['Energy Supply per Capita']
    Top15['Citable docs per Capita'] = Top15['Citable documents'] / Top15['PopEst']
    # Кореляція між документами на душу населення та енергією на душу населення
    return float(Top15['Citable docs per Capita'].corr(Top15['Energy Supply per Capita']))

def answer_seven():
    Top15 = answer_one()
    ContinentDict  = {'China':'Asia', 'United States':'North America', 'Japan':'Asia', 
                      'United Kingdom':'Europe', 'Russian Federation':'Europe', 
                      'Canada':'North America', 'Germany':'Europe', 'India':'Asia',
                      'France':'Europe', 'South Korea':'Asia', 'Italy':'Europe', 
                      'Spain':'Europe', 'Iran':'Asia', 'Australia':'Australia', 'Brazil':'South America'}
    
    Top15['PopEst'] = (Top15['Energy Supply'] / Top15['Energy Supply per Capita']).astype(float)
    Top15['Continent'] = [ContinentDict[country] for country in Top15.index]
    
    return Top15.groupby('Continent')['PopEst'].agg(['size', 'sum', 'mean', 'std'])

# ПЕРЕВІРКА ВСІХ ВІДПОВІДЕЙ
if __name__ == "__main__":
    print(f"Q1 (Shape): {answer_one().shape}")
    print(f"Q2 (avgGDP Top 3):\n{answer_two().head(3)}")
    print(f"Q3 (GDP Change): {answer_three()}")
    print(f"Q4 (Max Self-Cite): {answer_four()}")
    print(f"Q5 (3rd Populous): {answer_five()}")
    print(f"Q6 (Correlation): {answer_six()}")
    print(f"Q7 (Continents):\n{answer_seven()}")

Q1 (Shape): (15, 20)
Q2 (avgGDP Top 3):
Country
United States    1.570403e+13
China            6.927707e+12
Japan            5.239642e+12
Name: avgGDP, dtype: float64
Q3 (GDP Change): 118652421857.7959
Q4 (Max Self-Cite): ('China', np.float64(0.6912289816173135))
Q5 (3rd Populous): United States
Q6 (Correlation): 0.7434709127726777
Q7 (Continents):
               size           sum          mean           std
Continent                                                    
Asia              5  2.898666e+09  5.797333e+08  6.790979e+08
Australia         1  2.331602e+07  2.331602e+07           NaN
Europe            6  4.579297e+08  7.632161e+07  3.464767e+07
North America     2  3.528552e+08  1.764276e+08  1.996696e+08
South America     1  2.059153e+08  2.059153e+08           NaN
